# 01 – Tiền xử lý dữ liệu: kiểm tra, EDA, chia tập, chuẩn hóa, augmentation
**Người phụ trách:** Thành viên A &nbsp;|&nbsp; **Requirement:** Req 1 – Preprocess the data: clean, normalize, and augment

**Mục tiêu:** Chứng minh dữ liệu sạch, mô tả dữ liệu, tạo cách chia train/val/test dùng chung, minh họa chuẩn hóa và augmentation.

**Cần có trước:** Không – đây là notebook đầu tiên.

**File tạo ra:** `data/splits.csv`, `results/data_check.csv`, `results/split_summary.csv`, `figures/class_distribution.png`, `figures/sample_grid.png`, `figures/normalization.png`, `figures/augmented_samples.png`

> Chạy lần lượt từng cell từ trên xuống (`Shift + Enter`). Ô có chữ **✍️ NHẬN XÉT** là chỗ bạn tự viết phân tích.

## 0. Thiết lập môi trường
Cell này làm 3 việc:
1. **Trên Colab:** kết nối Google Drive (để lưu model và kết quả lâu dài) và tải bộ dữ liệu EuroSAT (~95 MB) về máy Colab.
2. **Trên máy cá nhân:** dùng luôn thư mục dự án (notebook nằm trong `notebooks/`, ảnh nằm trong `data/EuroSAT_RGB/`).
3. Tạo sẵn các thư mục lưu kết quả: `data/`, `models/`, `results/`, `figures/`.

> Trên Colab nhớ bật GPU trước: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
# ==============================================================================
# 0. THIẾT LẬP MÔI TRƯỜNG – luôn chạy cell này đầu tiên
# ==============================================================================
import os
import sys

IN_COLAB = "google.colab" in sys.modules                 # True nếu đang chạy trên Google Colab

if IN_COLAB:
    from google.colab import drive                        # thư viện kết nối Google Drive của Colab
    drive.mount("/content/drive")                         # hiện hộp thoại xin quyền truy cập Drive
    PROJECT_DIR = "/content/drive/MyDrive/eurosat_project"   # thư mục Drive DÙNG CHUNG của cả nhóm
    DATA_DIR = "/content/EuroSAT_RGB"                     # ảnh gốc để trên máy Colab (không lưu vào Drive)
    if not os.path.exists(DATA_DIR):                      # chỉ tải 1 lần cho mỗi phiên Colab
        os.system('wget -q -O /content/EuroSAT_RGB.zip "https://zenodo.org/records/7711810/files/EuroSAT_RGB.zip?download=1"')
        os.system("unzip -q -o /content/EuroSAT_RGB.zip -d /content/")   # giải nén -> /content/EuroSAT_RGB/
else:
    PROJECT_DIR = os.path.abspath("..")                   # thư mục gốc của dự án (cha của notebooks/)
    DATA_DIR = os.path.join(PROJECT_DIR, "data", "EuroSAT_RGB")

# Đường dẫn các file/thư mục kết quả – giống hệt cấu trúc repo GitHub
SPLITS_CSV  = os.path.join(PROJECT_DIR, "data", "splits.csv")   # cách chia train/val/test dùng chung
MODELS_DIR  = os.path.join(PROJECT_DIR, "models")               # model đã train (.keras)
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")              # bảng số liệu (.csv, .json, .txt)
FIGURES_DIR = os.path.join(PROJECT_DIR, "figures")              # hình vẽ (.png)
for folder in [os.path.dirname(SPLITS_CSV), MODELS_DIR, RESULTS_DIR, FIGURES_DIR]:
    os.makedirs(folder, exist_ok=True)                    # tạo thư mục nếu chưa có

assert os.path.exists(DATA_DIR), f"Không tìm thấy ảnh tại {DATA_DIR} – kiểm tra lại bước tải/giải nén dữ liệu"
print("Ảnh gốc       :", DATA_DIR)
print("Số thư mục lớp:", len([d for d in os.listdir(DATA_DIR) if not d.startswith(".")]))
print("Lưu kết quả tại:", PROJECT_DIR)

In [ ]:
# ==============================================================================
# 1. IMPORT THƯ VIỆN VÀ HẰNG SỐ DÙNG CHUNG
# ==============================================================================
import json                                   # đọc/ghi file .json (tóm tắt kết quả train)
import time                                   # đo thời gian train
import numpy as np                            # xử lý mảng số
import pandas as pd                           # xử lý bảng dữ liệu
import matplotlib.pyplot as plt               # vẽ hình
import seaborn as sns                         # vẽ heatmap đẹp hơn
from PIL import Image                         # đọc file ảnh .jpg

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import RandomFlip, RandomBrightness

RANDOM_STATE = 42                             # seed cố định để kết quả lặp lại được
tf.keras.utils.set_random_seed(RANDOM_STATE)  # cố định seed cho numpy, python và tensorflow

CLASS_NAMES = ["AnnualCrop", "Forest", "HerbaceousVegetation", "Highway", "Industrial",
               "Pasture", "PermanentCrop", "Residential", "River", "SeaLake"]   # 10 lớp, theo thứ tự label_id 0..9
NUM_CLASSES = len(CLASS_NAMES)                # = 10
IMG_SIZE = 64                                 # ảnh EuroSAT có kích thước 64 x 64
BATCH_SIZE = 64                               # số ảnh mỗi lần cập nhật trọng số

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "KHÔNG CÓ GPU – vào Runtime → Change runtime type → T4 GPU")

## 2. Kiểm tra dữ liệu (data cleaning)
Mở **từng ảnh** để tìm 4 loại lỗi:
1. **Ảnh hỏng** – không mở được.
2. **Sai kích thước** – khác 64×64.
3. **Không phải ảnh màu RGB** – vd. ảnh xám.
4. **Ảnh trùng lặp** – hai file có nội dung giống hệt nhau (so bằng mã băm MD5).

Ảnh lỗi sẽ bị loại ở bước chia tập (mục 4).

In [ ]:
# ==============================================================================
# 2. KIỂM TRA DỮ LIỆU
# ==============================================================================
import hashlib                                          # tính mã băm MD5 để phát hiện ảnh trùng

rows = []                                               # mỗi ảnh là 1 dòng thông tin
for cls in CLASS_NAMES:                                 # duyệt qua 10 thư mục lớp
    for fname in sorted(os.listdir(os.path.join(DATA_DIR, cls))):   # sắp xếp để thứ tự luôn cố định
        path = os.path.join(DATA_DIR, cls, fname)
        try:
            with Image.open(path) as im:
                im.verify()                             # kiểm tra file có bị hỏng không
            with Image.open(path) as im:
                width, height = im.size                 # kích thước ảnh
                mode = im.mode                          # kiểu màu: "RGB", "L" (xám)...
            readable = True
        except Exception:                               # không mở được -> ảnh hỏng
            width = height = mode = None
            readable = False
        with open(path, "rb") as f:
            md5 = hashlib.md5(f.read()).hexdigest()     # "dấu vân tay" của nội dung file
        rows.append({"filepath": f"{cls}/{fname}", "label": cls, "readable": readable,
                     "width": width, "height": height, "mode": mode, "md5": md5})

df = pd.DataFrame(rows)                                 # bảng thông tin của toàn bộ ảnh
df["bad_size"]  = df["readable"] & ((df["width"] != IMG_SIZE) | (df["height"] != IMG_SIZE))
df["bad_mode"]  = df["readable"] & (df["mode"] != "RGB")
df["duplicate"] = df.duplicated("md5", keep="first")    # True nếu nội dung đã xuất hiện trước đó

check = pd.DataFrame({
    "Tổng số ảnh":             [len(df)],
    "Ảnh hỏng":                [int((~df["readable"]).sum())],
    "Sai kích thước":          [int(df["bad_size"].sum())],
    "Không phải RGB":          [int(df["bad_mode"].sum())],
    "Ảnh trùng lặp":           [int(df["duplicate"].sum())],
})
check.to_csv(os.path.join(RESULTS_DIR, "data_check.csv"), index=False)
check.T

**✍️ NHẬN XÉT:** _Tổng cộng bao nhiêu ảnh? Có ảnh hỏng / sai kích thước / trùng lặp không? Nếu có thì loại bao nhiêu ảnh?_

## 3. Khám phá dữ liệu (EDA)
Xem số ảnh mỗi lớp (có mất cân bằng không) và vài ảnh mẫu của từng lớp.

In [ ]:
# ==============================================================================
# 3a. SỐ ẢNH MỖI LỚP
# ==============================================================================
counts = df[df["readable"]]["label"].value_counts().reindex(CLASS_NAMES)   # đếm số ảnh theo lớp
print(counts)
print("Tỷ lệ lớp nhiều nhất / ít nhất:", round(counts.max() / counts.min(), 2))

plt.figure(figsize=(10, 4))
counts.plot(kind="bar", color="steelblue")
plt.ylabel("Số ảnh"); plt.title("Số ảnh mỗi lớp – EuroSAT RGB")
plt.xticks(rotation=45, ha="right"); plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "class_distribution.png"), dpi=150)
plt.show()

In [ ]:
# ==============================================================================
# 3b. ẢNH MẪU: mỗi lớp 5 ảnh ngẫu nhiên
# ==============================================================================
fig, axes = plt.subplots(NUM_CLASSES, 5, figsize=(10, 20))
for r, cls in enumerate(CLASS_NAMES):
    samples = df[(df["label"] == cls) & df["readable"]]["filepath"].sample(5, random_state=RANDOM_STATE)
    for c, fp in enumerate(samples):
        axes[r, c].imshow(Image.open(os.path.join(DATA_DIR, fp)))
        axes[r, c].axis("off")
    axes[r, 0].set_title(cls, loc="left", fontsize=10)      # tên lớp ở đầu mỗi hàng
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "sample_grid.png"), dpi=120)
plt.show()

**✍️ NHẬN XÉT:** _Dữ liệu mất cân bằng nhiều hay ít? Những lớp nào nhìn bằng mắt thường đã thấy giống nhau (vd. Pasture – HerbaceousVegetation, Highway – River)?_

## 4. Chia train / validation / test = 70 / 15 / 15
- **Train:** để model học. **Validation:** để chọn epoch, chọn biến thể. **Test:** chỉ dùng **1 lần** ở notebook 05.
- **Stratified:** tỷ lệ mỗi lớp ở 3 tập giống nhau.
- Kết quả lưu vào `data/splits.csv` – **cả nhóm dùng chung**, không ai tự chia lại.

In [ ]:
# ==============================================================================
# 4. CHIA TRAIN / VALIDATION / TEST
# ==============================================================================
from sklearn.model_selection import train_test_split

# Chỉ giữ ảnh không lỗi
clean = df[df["readable"] & ~df["bad_size"] & ~df["bad_mode"] & ~df["duplicate"]].copy()
clean["label_id"] = clean["label"].map({name: i for i, name in enumerate(CLASS_NAMES)})   # tên lớp -> số 0..9

# Bước 1: tách 70% train, 30% còn lại
train_df, temp_df = train_test_split(clean, test_size=0.30, stratify=clean["label"],
                                     random_state=RANDOM_STATE)
# Bước 2: chia đôi 30% còn lại -> 15% validation, 15% test
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"],
                                   random_state=RANDOM_STATE)

train_df["split"], val_df["split"], test_df["split"] = "train", "val", "test"
splits = pd.concat([train_df, val_df, test_df])[["filepath", "label", "label_id", "split"]]
splits.to_csv(SPLITS_CSV, index=False)                     # file dùng chung cho cả nhóm

summary = pd.crosstab(splits["label"], splits["split"])[["train", "val", "test"]]   # số ảnh mỗi lớp x mỗi tập
summary.loc["TỔNG"] = summary.sum()
summary.to_csv(os.path.join(RESULTS_DIR, "split_summary.csv"))
summary

**✍️ NHẬN XÉT:** _Mỗi tập có bao nhiêu ảnh? Tỷ lệ các lớp giữa 3 tập có giống nhau không?_

## 5. Chuẩn hóa (normalization)
Pixel gốc nằm trong khoảng **[0, 255]**. Chia cho 255 để đưa về **[0, 1]** giúp model học ổn định hơn.
Ở các notebook 02–05, bước này nằm ngay trong cell nạp dữ liệu (`X = X / 255.0`).

In [ ]:
# ==============================================================================
# 5. MINH HỌA CHUẨN HÓA
# ==============================================================================
sample_paths = train_df["filepath"].values[:500]                                  # lấy thử 500 ảnh train
X_sample = np.stack([np.array(Image.open(os.path.join(DATA_DIR, fp))) for fp in sample_paths])
X_sample_norm = X_sample.astype("float32") / 255.0                               # chuẩn hóa về [0, 1]

print("Shape tensor ảnh:", X_sample.shape, "-> (số ảnh, cao, rộng, số kênh màu)")
print(f"Trước chuẩn hóa: min = {X_sample.min()}, max = {X_sample.max()}, kiểu = {X_sample.dtype}")
print(f"Sau chuẩn hóa  : min = {X_sample_norm.min():.2f}, max = {X_sample_norm.max():.2f}, kiểu = {X_sample_norm.dtype}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(X_sample.ravel(), bins=50, color="gray"); axes[0].set_title("Pixel trước chuẩn hóa [0, 255]")
axes[1].hist(X_sample_norm.ravel(), bins=50, color="steelblue"); axes[1].set_title("Pixel sau chuẩn hóa [0, 1]")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "normalization.png"), dpi=150)
plt.show()

## 3. Data augmentation (chỉ hoạt động khi train)
Dùng các **lớp tiền xử lý của Keras** đặt ở đầu model, nên augmentation **tự động tắt** khi đánh giá/dự đoán.
- `RandomFlip("horizontal_and_vertical")`: lật ngang / dọc ngẫu nhiên. Ảnh vệ tinh chụp từ trên xuống, không có hướng "lên trên" nên lật không làm đổi ý nghĩa.
- `RandomBrightness(0.1)`: tăng/giảm độ sáng tối đa 10% – mô phỏng khác biệt ánh sáng, khí quyển.
- **Không dùng:** zoom mạnh (mỗi pixel = 10 m cố định, zoom làm sai kích thước ruộng, nhà, đường), xoay góc lẻ (tạo góc đen), đổi màu (màu nước, cây mang ý nghĩa).

> Augmentation giống hệt nhau ở cả 3 model để so sánh công bằng.

In [ ]:
# ==============================================================================
# 6. DATA AUGMENTATION – giống hệt nhau ở cả 3 model
# ==============================================================================
data_augmentation = Sequential([
    RandomFlip("horizontal_and_vertical"),             # lật ngang và/hoặc lật dọc ngẫu nhiên
    RandomBrightness(factor=0.1, value_range=(0, 1)),  # đổi độ sáng ±10% (ảnh đã ở thang 0–1)
], name="data_augmentation")

In [ ]:
# Xem thử: 1 ảnh gốc (cột đầu) và 5 phiên bản sau augmentation
fig, axes = plt.subplots(4, 6, figsize=(12, 8))
for r in range(4):
    img = X_sample_norm[r * 50]                                         # chọn 1 ảnh
    axes[r, 0].imshow(img); axes[r, 0].set_title("Ảnh gốc", fontsize=9)
    for c in range(1, 6):
        aug = data_augmentation(img[None, ...], training=True)[0]       # training=True để bật augmentation
        axes[r, c].imshow(np.clip(np.asarray(aug), 0, 1)); axes[r, c].set_title(f"Augment {c}", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "augmented_samples.png"), dpi=120)
plt.show()

**✍️ NHẬN XÉT:** _Vì sao chọn lật ngang/dọc và đổi độ sáng nhẹ? Vì sao KHÔNG dùng zoom mạnh, xoay góc lẻ, đổi màu cho ảnh vệ tinh?_

## ✅ Xong notebook 01
Kiểm tra trong thư mục dự án đã có `data/splits.csv` → báo nhóm để B và C bắt đầu train.